<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/midterm/MID1_imitation_game_lab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Machines, Tests and Meaning

## Project 1: An imitation game laboratory

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

Two machine witnesses, a pattern-matching program in the style of ELIZA and a Markov chain trained on a small corpus, play three-party games against constructed human replies. A learned interrogator judges the replies from surface cues, and the results are analysed with exact power calculations and Bayesian estimates of the win rate [1, 2, 3].

**Required extensions.** Add a stronger witness, for example a retrieval-based responder or a small language model, and at least two further cues or a text classifier as interrogator. Determine how many games are needed to test Turing's 70 percent criterion reliably, and argue in the report what the win rates can and cannot show, with reference to Blockhead and to subcognitive questions [4, 5].

## What this example implements

The notebook builds two machine witnesses, extracts seven surface features from replies, trains a logistic interrogator, simulates judges with different attentiveness, and computes the power of exact binomial tests and a Beta posterior for an observed win rate.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Witnesses and cues

In [ ]:
import re
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from scipy.stats import binom, beta as beta_dist

rng = np.random.default_rng(SEED)
QUESTIONS = ["What did you do last weekend?", "What is your favourite food and why?", "Describe the room you are in.",
             "What annoys you most about public transport?", "Tell me about a mistake you made recently.",
             "What would you do with a free afternoon?", "How do you feel about rainy days?", "What was the last film you watched?"]
HUMAN = ["went hiking with friends, legs still hurt lol", "honestly pizza, nothing complicated", "small desk, messy, a plant that is slowly dying",
         "buses never on time. never.", "sent an email to the wrong person, awkward", "nap. definitely a nap", "i like them tbh, cozy weather",
         "some thriller, forgot the name already", "stayed home mostly, cleaned the kitchen", "lentil soup my grandmother makes",
         "window, bookshelf, too many cables", "people talking loudly on the phone", "forgot my friend's birthday... again",
         "walk by the lake maybe", "depends, if i dont have to go out its fine", "a documentary about octopuses, weirdly good"]
REFLECT = {"i": "you", "my": "your", "me": "you", "you": "I", "your": "my", "am": "are"}

def eliza(q):
    words = [REFLECT.get(w, w) for w in re.findall(r"[a-z']+", q.lower())]
    t = ["Why do you ask about {}?", "What makes you think of {}?", "Does it matter to you {}?", "Tell me more about {}."]
    return t[rng.integers(len(t))].format(" ".join(words[-4:]))

CORPUS = ("i went to the market and bought some bread . the weather was nice so i walked home . i like films about space and old music . "
          "my friend called me and we talked about work . the bus was late again and i was annoyed . i made soup for dinner and watched a film . "
          "on weekends i read books and sometimes i go hiking . the room is small but it has a big window .").split()
BIGRAMS = defaultdict(list)
for a, b in zip(CORPUS, CORPUS[1:]):
    BIGRAMS[a].append(b)

def markov(q, length=12):
    w = str(rng.choice(["i", "the", "my", "on"])); out = [w]
    for _ in range(length - 1):
        w = str(rng.choice(BIGRAMS[w])) if BIGRAMS[w] else str(rng.choice(CORPUS)); out.append(w)
    return " ".join(out).replace(" .", ".")

def features(q, reply):
    toks = re.findall(r"[a-z']+", reply.lower()); qt = set(re.findall(r"[a-z']+", q.lower())); n = max(len(toks), 1)
    return [len(toks), len(set(toks)) / n, float(reply.strip().endswith("?")), sum(t in qt for t in toks) / n,
            float(reply[:1].islower()), float(bool(re.search(r"lol|tbh|honestly|\.\.\.", reply))), sum(t in ("i", "my", "me") for t in toks) / n]

for q in QUESTIONS[:2]:
    print(q, "| ELIZA:", eliza(q), "| Markov:", markov(q))

## 2. A learned interrogator and win rates

In [ ]:
def games(n, machine):
    X, y = [], []
    for _ in range(n):
        q = str(rng.choice(QUESTIONS))
        X.append(features(q, str(rng.choice(HUMAN)))); y.append(1)
        X.append(features(q, machine(q))); y.append(0)
    return np.array(X), np.array(y)

judge_X, judge_y = map(np.concatenate, zip(games(200, eliza), games(200, markov)))
judge = LogisticRegression(max_iter=2000).fit(judge_X, judge_y)

def win_rate(machine, temperature, n=400):
    wins = 0
    for _ in range(n):
        q = str(rng.choice(QUESTIONS))
        s_h = judge.decision_function([features(q, str(rng.choice(HUMAN)))])[0]
        s_m = judge.decision_function([features(q, machine(q))])[0]
        wins += rng.random() < 1 / (1 + np.exp(-(s_m - s_h) / temperature))
    return wins / n

for name, m in [("ELIZA", eliza), ("Markov chain", markov)]:
    print(name, {f"judge temperature {t}": round(win_rate(m, t), 3) for t in [0.5, 2.0, 8.0]})

## 3. How many games are enough?

In [ ]:
def power(n, p_true, alpha=0.05):
    k = np.arange(n + 1)
    pv = np.array([min(1.0, 2 * min(binom.cdf(x, n, 0.5), binom.sf(x - 1, n, 0.5))) for x in k])
    return float(binom.pmf(k[pv < alpha], n, p_true).sum())

ns = [10, 20, 40, 80, 160, 320]
plt.plot(ns, [power(n, 0.3) for n in ns], "o-", label="true win rate 0.30")
plt.plot(ns, [power(n, 0.4) for n in ns], "s-", label="true win rate 0.40")
plt.xscale("log"); plt.xlabel("number of games"); plt.ylabel("power of an exact binomial test against 0.5"); plt.legend(); plt.show()
k, n = 37, 100
post = beta_dist(1 + k, 1 + n - k)
print(f"{k} wins in {n} games: posterior mean {post.mean():.3f}, P(win rate > 0.5) = {post.sf(0.5):.4f}, 95% interval {post.ppf([0.025, 0.975]).round(3)}")

## Report and submission

Both tracks follow the course report template. A research report states its thesis in the introduction, reconstructs the relevant arguments as explicit premises and conclusions, and evaluates them. An application report explains what the simulation or experiment can and cannot show about the philosophical question, presents its results with figures and discusses its limitations. All sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Turing, A. M. (1950). Computing machinery and intelligence. *Mind*, 59(236), 433-460. <https://doi.org/10.1093/mind/LIX.236.433>

[2] Weizenbaum, J. (1966). ELIZA: A computer program for the study of natural language communication between man and machine. *Communications of the ACM*, 9(1), 36-45. <https://doi.org/10.1145/365153.365168>

[3] Jones, C. R., & Bergen, B. K. (2025). Large language models pass the Turing test. arXiv preprint arXiv:2503.23674. <https://arxiv.org/abs/2503.23674>

[4] Block, N. (1981). Psychologism and behaviorism. *The Philosophical Review*, 90(1), 5-43. <https://doi.org/10.2307/2184371>

[5] French, R. M. (1990). Subcognition and the limits of the Turing test. *Mind*, 99(393), 53-65. <https://doi.org/10.1093/mind/XCIX.393.53>